In [ ]:
import nibabel as nib
import numpy as np
from pathlib import Path
from typing import Tuple, Optional, Dict

class BrainVolumeCalculator:
    """
    A calculator for processing 3D MRI segmentation masks to compute
    absolute volumes and relative volume ratios (occupancy rates).
    """

    def __init__(self, target_label_idx: int = 1):
        """
        Initialize the calculator.

        Args:
            target_label_idx (int): The integer label ID for the target structure
                                    (e.g., 1 for Hippocampus). Defaults to 1.
        """
        self.target_label_idx = target_label_idx

    def load_nifti(self, file_path: str) -> Tuple[np.ndarray, Tuple[float, float, float]]:
        """
        Loads a NIfTI file and extracts data and voxel dimensions.

        Args:
            file_path (str): Path to the NIfTI file (.nii or .nii.gz).

        Returns:
            Tuple[np.ndarray, Tuple[float, float, float]]:
                - Image data array
                - Voxel dimensions (x, y, z) in mm
        """
        img = nib.load(file_path)
        data = img.get_fdata()
        header = img.header
        # Get voxel dimensions (zooms) usually in mm
        voxel_dims = header.get_zooms()[:3]
        return data, voxel_dims

    def calculate_metrics(self, mask_path: str, reference_mask_path: Optional[str] = None) -> Dict[str, float]:
        """
        Calculates absolute volume and volume ratio.

        Args:
            mask_path (str): Path to the segmentation mask (ROI).
            reference_mask_path (Optional[str]): Path to the reference mask (e.g., Total Intracranial Volume)
                                                 for ratio calculation. If None, ratio is skipped.

        Returns:
            Dict[str, float]: Dictionary containing:
                - 'voxel_count': Raw count of voxels
                - 'voxel_volume_mm3': Volume of a single voxel
                - 'absolute_volume_mm3': Total volume in cubic millimeters
                - 'volume_ratio_percent': (Optional) Percentage of ROI relative to reference
        """
        # Load Target Mask
        data, voxel_dims = self.load_nifti(mask_path)

        # Calculate volume of a single voxel (x * y * z)
        voxel_vol = np.prod(voxel_dims)

        # Count target voxels
        target_voxel_count = np.sum(data == self.target_label_idx)

        # Calculate Absolute Volume
        absolute_vol = target_voxel_count * voxel_vol

        results = {
            "voxel_count": int(target_voxel_count),
            "voxel_volume_mm3": float(voxel_vol),
            "absolute_volume_mm3": float(absolute_vol)
        }

        # Calculate Ratio if reference is provided (e.g., TIV)
        if reference_mask_path:
            ref_data, _ = self.load_nifti(reference_mask_path)
            # Assuming reference mask is binary or we count all non-zero labels as the reference volume
            ref_voxel_count = np.count_nonzero(ref_data)

            if ref_voxel_count > 0:
                ref_vol = ref_voxel_count * voxel_vol
                ratio = (absolute_vol / ref_vol) * 100
                results["reference_volume_mm3"] = float(ref_vol)
                results["volume_ratio_percent"] = float(ratio)
            else:
                results["volume_ratio_percent"] = 0.0

        return results


from metrics import BrainVolumeCalculator
import os

def main():
    # 설정: 해마(Hippocampus) 라벨이 1번이라고 가정
    calculator = BrainVolumeCalculator(target_label_idx=1)

    # 파일 경로 설정 (예시)
    sample_mask = "data/predictions/hippocampus_pred_001.nii.gz"
    sample_tiv = "data/masks/tiv_001.nii.gz"  # 전체 뇌 용적(참조용)

    if os.path.exists(sample_mask):
        print(f"Processing: {sample_mask}...")

        # 1. 절대 부피만 계산할 경우
        # results = calculator.calculate_metrics(sample_mask)

        # 2. 용적률(Ratio)까지 계산할 경우 (TIV 파일 필요)
        results = calculator.calculate_metrics(mask_path=sample_mask, reference_mask_path=sample_tiv)

        # 결과 출력 (소수점 포맷팅)
        print("-" * 30)
        print(f"Voxel Count      : {results['voxel_count']:,} ea")
        print(f"Absolute Volume  : {results['absolute_volume_mm3']:.2f} mm³")

        if 'volume_ratio_percent' in results:
            print(f"Volume Ratio     : {results['volume_ratio_percent']:.4f} % (Target / Reference)")
        print("-" * 30)

    else:
        print("Error: Input file not found.")

if __name__ == "__main__":
    main()